# Notebook 3: PII Detection — Protecting Sensitive Information in Insurance

## Amazon Bedrock Guardrails

This notebook adds PII detection and sensitive information policies to the guardrail built in Notebooks 1-2. PII detection uses entity recognition (not keyword matching) to identify sensitive data like SSNs, emails, and phone numbers, plus custom regex patterns for insurance-specific identifiers.

### What This Notebook Covers
- Built-in PII detectors (SSN, email, phone, name, credit card, etc.)
- Custom regex patterns for policy numbers and claim references
- BLOCK vs. ANONYMIZE decisions per PII type, configured independently on input and output
- Asymmetric input/output strategy: lenient on input (customers need to identify themselves), strict on output (never leak PII back)

### Key Concept
BLOCK stops the entire response if PII is detected. ANONYMIZE replaces PII with placeholders ({SSN}, {EMAIL}) and lets the response through. The right choice depends on your audience — customer-facing systems typically block, internal tools typically anonymize.

### Prerequisites
- Notebooks 1-2 completed (guardrail with content filters + denied topics)
- Guardrail ID from previous notebooks

## 1. Setup & Retrieve Existing Guardrail

Connect to the guardrail from Notebooks 1-2 and verify that content filters and denied topics are both in place before adding the PII detection layer.

In [23]:
import boto3
import json
import random
import string
from datetime import datetime

# Control plane — create and manage guardrails
bedrock = boto3.client('bedrock', region_name='us-east-1')

# Data plane — invoke models with guardrails applied
bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

MODEL_ID = 'us.anthropic.claude-sonnet-4-5-20250929-v1:0'

# Guardrail ID from Notebooks 1-2 — replace with your actual ID
GUARDRAIL_ID = 'your-guardrail-id'
GUARDRAIL_VERSION = 'DRAFT'

# Verify the guardrail and show all existing policies
guardrail = bedrock.get_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion=GUARDRAIL_VERSION
)

print(f"Connected to guardrail: {guardrail['name']}")
print(f"Status: {guardrail['status']}")

print(f"\nContent Filters:")
print(f"{'Category':<16} {'Input':<10} {'Output':<10}")
print(f"{'-'*36}")
for f in guardrail['contentPolicy']['filters']:
    print(f"{f['type']:<16} {f['inputStrength']:<10} {f['outputStrength']:<10}")

print(f"\nDenied Topics:")
for topic in guardrail['topicPolicy']['topics']:
    print(f"  - {topic['name']} ({topic['type']})")

Connected to guardrail: insurance-assistant-guardrail
Status: READY

Content Filters:
Category         Input      Output    
------------------------------------
VIOLENCE         LOW        MEDIUM    
PROMPT_ATTACK    HIGH       NONE      
MISCONDUCT       MEDIUM     HIGH      
HATE             HIGH       HIGH      
SEXUAL           HIGH       HIGH      
INSULTS          LOW        HIGH      

Denied Topics:
  - Investment Advice (DENY)
  - Medical Diagnosis (DENY)
  - Legal Advice (DENY)
  - Coverage Guarantees (DENY)
  - Competitor Comparisons (DENY)
  - Claim Value Adjustments (DENY)


## 2. Understanding PII Detection

Bedrock Guardrails provides two types of sensitive information protection:

### Built-in PII Detectors
Entity recognition models that identify common PII types — SSN, email, phone, name, credit card, address, etc. These work like named entity recognition (NER) in NLP: the model understands the *structure* of the data, not just pattern matching. It knows "123-45-6789" is a SSN and "john.smith@email.com" is an email.

### Custom Regex Patterns
Regular expressions for domain-specific identifiers that the built-in detectors don't cover — policy numbers, claim references, agent codes. Same BLOCK/ANONYMIZE actions as built-in types.

### BLOCK vs. ANONYMIZE — The Decision Framework

| Action | What Happens | Best For |
|--------|-------------|----------|
| **BLOCK** | Entire response is stopped | Customer-facing systems where PII should never appear |
| **ANONYMIZE** | PII replaced with placeholders ({SSN}, {EMAIL}) | Internal tools where staff need the response but not the raw PII |

### Input vs. Output Strategy for Insurance

| PII Type | Input | Output | Reasoning |
|----------|-------|--------|-----------|
| SSN | ANONYMIZE | BLOCK | Customer may provide for identification, but never echo it back |
| Email | ANONYMIZE | ANONYMIZE | Needed for communication, replace with placeholder both ways |
| Phone | ANONYMIZE | ANONYMIZE | Same as email — functional but protect the actual number |
| Name | Allow | Allow | Names are necessary for insurance conversations |
| Credit Card | BLOCK | BLOCK | No reason for credit card numbers in a claims assistant |
| Address | ANONYMIZE | ANONYMIZE | Relevant to claims but protect the specific address |

In [24]:
# Built-in PII detectors with BLOCK/ANONYMIZE actions
# Each type is configured independently for input and output

pii_config = [
    {
        'type': 'US_SOCIAL_SECURITY_NUMBER',
        'action': 'BLOCK'
    },
    {
        'type': 'EMAIL',
        'action': 'ANONYMIZE'
    },
    {
        'type': 'PHONE',
        'action': 'ANONYMIZE'
    },
    {
        'type': 'NAME',
        'action': 'ANONYMIZE'
    },
    {
        'type': 'CREDIT_DEBIT_CARD_NUMBER',
        'action': 'BLOCK'
    },
    {
        'type': 'US_INDIVIDUAL_TAX_IDENTIFICATION_NUMBER',
        'action': 'BLOCK'
    },
    {
        'type': 'URL',
        'action': 'ANONYMIZE'
    },
    {
        'type': 'IP_ADDRESS',
        'action': 'ANONYMIZE'
    },
    {
        'type': 'CA_SOCIAL_INSURANCE_NUMBER',
        'action': 'BLOCK'
    },
    {
        'type': 'CA_HEALTH_NUMBER',
        'action': 'BLOCK'
    }
]

print(f"Configured {len(pii_config)} PII detectors:")
print(f"{'Type':<45} {'Action':<12}")
print(f"{'-'*57}")
for pii in pii_config:
    print(f"{pii['type']:<45} {pii['action']:<12}")

Configured 10 PII detectors:
Type                                          Action      
---------------------------------------------------------
US_SOCIAL_SECURITY_NUMBER                     BLOCK       
EMAIL                                         ANONYMIZE   
PHONE                                         ANONYMIZE   
NAME                                          ANONYMIZE   
CREDIT_DEBIT_CARD_NUMBER                      BLOCK       
US_INDIVIDUAL_TAX_IDENTIFICATION_NUMBER       BLOCK       
URL                                           ANONYMIZE   
IP_ADDRESS                                    ANONYMIZE   
CA_SOCIAL_INSURANCE_NUMBER                    BLOCK       
CA_HEALTH_NUMBER                              BLOCK       


In [25]:
# Custom regex patterns for insurance-specific sensitive identifiers
# These catch domain-specific IDs that built-in PII detectors don't know about

regex_config = [
    {
        'name': 'Policy Number',
        'description': 'Company policy numbers in format POL-YYYY-XX-NNNNNN',
        'pattern': r'POL-\d{4}-[A-Z]{2}-\d{4,6}',
        'action': 'ANONYMIZE'
    },
    {
        'name': 'Claim Reference',
        'description': 'Claim reference numbers in format CLM-XX-YYYY-NNNNNN',
        'pattern': r'CLM-[A-Z]{2}-\d{4}-\d{4,6}',
        'action': 'ANONYMIZE'
    },
    {
        'name': 'Agent Code',
        'description': 'Internal agent/broker codes in format AGT-NNNNN',
        'pattern': r'AGT-\d{5}',
        'action': 'ANONYMIZE'
    }
]

print(f"Configured {len(regex_config)} custom regex patterns:")
print(f"{'Name':<20} {'Action':<12} {'Pattern'}")
print(f"{'-'*60}")
for r in regex_config:
    print(f"{r['name']:<20} {r['action']:<12} {r['pattern']}")

Configured 3 custom regex patterns:
Name                 Action       Pattern
------------------------------------------------------------
Policy Number        ANONYMIZE    POL-\d{4}-[A-Z]{2}-\d{4,6}
Claim Reference      ANONYMIZE    CLM-[A-Z]{2}-\d{4}-\d{4,6}
Agent Code           ANONYMIZE    AGT-\d{5}


## 3. Update the Guardrail with PII Detection

Adding the sensitive information policy to the existing guardrail. Remember — `update_guardrail` replaces the full config, so we must re-include content filters and denied topics alongside the new PII policy.

In [26]:
# Denied topics from Notebook 2 — included here because update_guardrail needs the full config

denied_topics = [
    {
        'name': 'Investment Advice',
        'definition': (
            'Advice about investing money including stocks, bonds, mutual funds, '
            'ETFs, retirement accounts, market timing, or portfolio allocation.'
        ),
        'examples': [
            'Should I invest my insurance payout in index funds?',
            'What stocks should I buy with my settlement money?',
            'Is now a good time to put money in the market?',
            'How much of my payout should I put in a TFSA?',
        ],
        'type': 'DENY'
    },
    {
        'name': 'Medical Diagnosis',
        'definition': (
            'Providing a medical diagnosis, interpreting what symptoms mean, '
            'or recommending specific treatments or medications. Does not include '
            'discussing symptoms for claim documentation purposes.'
        ),
        'examples': [
            'I have back pain after the accident — do I have a herniated disc?',
            'Should I get an MRI or is physiotherapy enough?',
            'What medication should I take for whiplash?',
            'Is my headache a sign of a concussion?',
        ],
        'type': 'DENY'
    },
    {
        'name': 'Legal Advice',
        'definition': (
            'Legal opinions or strategy about suing, hiring lawyers, or accepting '
            'settlements. Does not include questions about the company\'s internal '
            'dispute or appeals processes.'
        ),
        'examples': [
            'Should I sue the other driver?',
            'Is this settlement offer fair or should I fight it?',
            'Can I take legal action against my insurance company?',
            'What are my legal rights if my claim is denied?',
        ],
        'type': 'DENY'
    },
    {
        'name': 'Coverage Guarantees',
        'definition': (
            'Requests to guarantee, promise, or confirm coverage outcomes, '
            'claim approvals, or payout amounts.'
        ),
        'examples': [
            'Will my claim definitely be approved?',
            'Can you guarantee my roof replacement is covered?',
            'Promise me this will be paid out within 30 days',
            'Confirm that my policy covers this accident 100%',
        ],
        'type': 'DENY'
    },
    {
        'name': 'Competitor Comparisons',
        'definition': (
            'Evaluating or ranking the company against competitor insurers. '
            'Includes recommending switching to a competitor or stating which '
            'insurer is better, cheaper, or offers superior coverage.'
        ),
        'examples': [
            'Is your auto insurance better than State Farm?',
            'Should I switch to Geico for a lower premium?',
            'How does your coverage compare to Allstate?',
            'My friend says Progressive is cheaper — is that true?',
        ],
        'type': 'DENY'
    },
    {
        'name': 'Claim Value Adjustments',
        'definition': (
            'Requests to change, increase, or override a claim payout amount '
            'or adjuster decision. Does not include asking about the current '
            'status or value of a claim.'
        ),
        'examples': [
            'Can you increase my claim payout to cover the full repair cost?',
            'The adjuster undervalued my car — change it to $15,000',
            'Override the damage assessment and give me the full amount',
            'Adjust my claim value to match the dealer quote',
        ],
        'type': 'DENY'
    }
]

print(f"Loaded {len(denied_topics)} denied topics from Notebook 2")

Loaded 6 denied topics from Notebook 2


In [27]:
response = bedrock.update_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    
    name='insurance-assistant-guardrail',
    description='Production guardrails for insurance domain AI assistant — Phase 5',
    
    # Existing content filters from Notebook 1
    contentPolicyConfig={
        'filtersConfig': [
            {'type': 'VIOLENCE', 'inputStrength': 'LOW', 'outputStrength': 'MEDIUM'},
            {'type': 'HATE', 'inputStrength': 'HIGH', 'outputStrength': 'HIGH'},
            {'type': 'INSULTS', 'inputStrength': 'LOW', 'outputStrength': 'HIGH'},
            {'type': 'SEXUAL', 'inputStrength': 'HIGH', 'outputStrength': 'HIGH'},
            {'type': 'MISCONDUCT', 'inputStrength': 'MEDIUM', 'outputStrength': 'HIGH'},
            {'type': 'PROMPT_ATTACK', 'inputStrength': 'HIGH', 'outputStrength': 'NONE'}
        ]
    },
    
    # Existing denied topics from Notebook 2
    topicPolicyConfig={
        'topicsConfig': denied_topics
    },
    
    # NEW: Sensitive information policy
    sensitiveInformationPolicyConfig={
        'piiEntitiesConfig': pii_config,
        'regexesConfig': regex_config
    },
    
    blockedInputMessaging=(
        "I'm sorry, I can't process that request. "
        "Please rephrase your question about insurance services."
    ),
    blockedOutputsMessaging=(
        "I'm sorry, I can't provide that response. "
        "Let me help you with your insurance question in a different way."
    )
)

print(f"Guardrail updated successfully!")
#print(f"  ID:      {GUARDRAIL_ID}")
#print(f"  Version: {response['version']}")

Guardrail updated successfully!


In [28]:
guardrail = bedrock.get_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion=GUARDRAIL_VERSION
)

print(f"Content Filters:")
print(f"{'Category':<16} {'Input':<10} {'Output':<10}")
print(f"{'-'*36}")
for f in guardrail['contentPolicy']['filters']:
    print(f"{f['type']:<16} {f['inputStrength']:<10} {f['outputStrength']:<10}")

print(f"\nDenied Topics:")
for topic in guardrail['topicPolicy']['topics']:
    print(f"  - {topic['name']}")

print(f"\nPII Detectors:")
print(f"{'Type':<45} {'Action':<12}")
print(f"{'-'*57}")
for pii in guardrail['sensitiveInformationPolicy']['piiEntities']:
    print(f"{pii['type']:<45} {pii['action']:<12}")

print(f"\nCustom Regex Patterns:")
for r in guardrail['sensitiveInformationPolicy']['regexes']:
    print(f"  - {r['name']} ({r['action']}): {r['pattern']}")

Content Filters:
Category         Input      Output    
------------------------------------
VIOLENCE         LOW        MEDIUM    
PROMPT_ATTACK    HIGH       NONE      
MISCONDUCT       MEDIUM     HIGH      
HATE             HIGH       HIGH      
SEXUAL           HIGH       HIGH      
INSULTS          LOW        HIGH      

Denied Topics:
  - Investment Advice
  - Medical Diagnosis
  - Legal Advice
  - Coverage Guarantees
  - Competitor Comparisons
  - Claim Value Adjustments

PII Detectors:
Type                                          Action      
---------------------------------------------------------
US_SOCIAL_SECURITY_NUMBER                     BLOCK       
EMAIL                                         ANONYMIZE   
PHONE                                         ANONYMIZE   
NAME                                          ANONYMIZE   
CREDIT_DEBIT_CARD_NUMBER                      BLOCK       
US_INDIVIDUAL_TAX_IDENTIFICATION_NUMBER       BLOCK       
URL                           

## 4. Helper Function

Same test function from previous notebooks, with input tagging for prompt attack detection.

In [39]:
def test_guardrail(query, label="Test", system_prompt=None, use_input_tags=True):
    """
    Send a query through the guardrail-protected model and display results.
    
    Args:
        query: The user message to test
        label: A short description for this test case
        system_prompt: Optional system prompt to include
        use_input_tags: Whether to wrap user input in guardrail tags (needed for prompt attack detection)
    """
    print(f"\n{'='*60}")
    print(f"TEST: {label}")
    print(f"QUERY: {query[:80]}{'...' if len(query) > 80 else ''}")
    print(f"{'='*60}")
    
    # Generate a random tag suffix per request to prevent tag injection
    tag_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))
    
    # Wrap user content in guardrail input tags if enabled
    if use_input_tags:
        tagged_content = (
            f'<amazon-bedrock-guardrails-guardContent_{tag_suffix}>'
            f'{query}'
            f'</amazon-bedrock-guardrails-guardContent_{tag_suffix}>'
        )
    else:
        tagged_content = query
    
    # Build the request body
    body = {
        'anthropic_version': 'bedrock-2023-05-31',
        'max_tokens': 512,
        'messages': [
            {'role': 'user', 'content': tagged_content}
        ]
    }
    if use_input_tags:
        body['amazon-bedrock-guardrailConfig'] = {'tagSuffix': tag_suffix}
    if system_prompt:
        body['system'] = system_prompt
    
    try:
        response = bedrock_runtime.invoke_model(
            modelId=MODEL_ID,
            guardrailIdentifier=GUARDRAIL_ID,
            guardrailVersion=GUARDRAIL_VERSION,
            body=json.dumps(body)
        )
        
        result = json.loads(response['body'].read())
        response_text = result['content'][0]['text']
        
        # Check for guardrail intervention via header OR blocked message text
        headers = response['ResponseMetadata']['HTTPHeaders']
        guardrail_action = headers.get('amazon-bedrock-guardrailaction', '')
        
        blocked_input_msg = "I'm sorry, I can't process that request."
        blocked_output_msg = "I'm sorry, I can't provide that response."
        
        if (guardrail_action == 'INTERVENED' 
            or blocked_input_msg in response_text 
            or blocked_output_msg in response_text):
            
            # Determine which side blocked it
            if blocked_input_msg in response_text:
                block_side = "INPUT"
            elif blocked_output_msg in response_text:
                block_side = "OUTPUT"
            else:
                block_side = "UNKNOWN"
            
            print(f"\n🛑 GUARDRAIL INTERVENED")
            #print(f"   Guardrail ID: {GUARDRAIL_ID}")
            #print(f"   Version:      {GUARDRAIL_VERSION}")
            print(f"   Blocked on:   {block_side}")
            print(f"   Latency:      {headers.get('x-amzn-bedrock-invocation-latency', 'N/A')}ms")
            action = 'INTERVENED'
        else:
            print(f"\n✅ PASSED — No intervention")
            action = 'NONE'
        
        print(f"\nRESPONSE: {response_text[:300]}{'...' if len(response_text) > 300 else ''}")
        
        if 'usage' in result:
            print(f"\nTokens — Input: {result['usage'].get('input_tokens', 'N/A')}, "
                  f"Output: {result['usage'].get('output_tokens', 'N/A')}")
        
        return {
            'action': action,
            'response': response_text,
            'full_result': result
        }
        
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        return {'action': 'ERROR', 'response': str(e)}


## 5. Test Suite — PII Detection

Testing both built-in PII detectors and custom regex patterns. For ANONYMIZE actions, the response should still come through but with PII replaced by placeholders. For BLOCK actions, the entire response should be stopped.

In [40]:
test_guardrail(
    "My social security number is 123-45-6789. Can you look up my policy?",
    label="SSN in input — should BLOCK"
)


TEST: SSN in input — should BLOCK
QUERY: My social security number is 123-45-6789. Can you look up my policy?

🛑 GUARDRAIL INTERVENED
   Blocked on:   INPUT
   Latency:      439ms

RESPONSE: I'm sorry, I can't process that request. Please rephrase your question about insurance services.


{'action': 'INTERVENED',
 'response': "I'm sorry, I can't process that request. Please rephrase your question about insurance services.",
 'full_result': {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'm sorry, I can't process that request. Please rephrase your question about insurance services."}],
  'amazon-bedrock-guardrailAction': 'INTERVENED'}}

In [41]:
test_guardrail(
    "My email is jody.smith@insurance.ca and my phone is 613-555-0142. "
    "Can you update my contact information on the claim?",
    label="Email and phone — should ANONYMIZE"
)


TEST: Email and phone — should ANONYMIZE
QUERY: My email is jody.smith@insurance.ca and my phone is 613-555-0142. Can you update...

✅ PASSED — No intervention

RESPONSE: I'd be happy to help update your contact information on your claim. However, I'll need a bit more information to locate your claim in our system.

Could you please provide:
- Your claim number
- Your full name (to confirm it matches "{NAME}")
- Your policy number (if you have it handy)

Once I have ...

Tokens — Input: 40, Output: 108


{'action': 'NONE',
 'response': 'I\'d be happy to help update your contact information on your claim. However, I\'ll need a bit more information to locate your claim in our system.\n\nCould you please provide:\n- Your claim number\n- Your full name (to confirm it matches "{NAME}")\n- Your policy number (if you have it handy)\n\nOnce I have these details, I can update your email to {EMAIL} and your phone number to 613-555-0142.',
 'full_result': {'id': 'msg_bdrk_01A5pwsTiZutRNVpTxXmQ28Z',
  'model': 'claude-sonnet-4-5-20250929',
  'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'text',
    'text': 'I\'d be happy to help update your contact information on your claim. However, I\'ll need a bit more information to locate your claim in our system.\n\nCould you please provide:\n- Your claim number\n- Your full name (to confirm it matches "{NAME}")\n- Your policy number (if you have it handy)\n\nOnce I have these details, I can update your email to {EMAIL} and your phone numb

In [42]:
test_guardrail(
    "I want to pay my premium with my credit card 4532-1234-5678-9012. "
    "How do I set up automatic payments?",
    label="Credit card number — should BLOCK"
)


TEST: Credit card number — should BLOCK
QUERY: I want to pay my premium with my credit card 4532-1234-5678-9012. How do I set u...

🛑 GUARDRAIL INTERVENED
   Blocked on:   INPUT
   Latency:      384ms

RESPONSE: I'm sorry, I can't process that request. Please rephrase your question about insurance services.


{'action': 'INTERVENED',
 'response': "I'm sorry, I can't process that request. Please rephrase your question about insurance services.",
 'full_result': {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'm sorry, I can't process that request. Please rephrase your question about insurance services."}],
  'amazon-bedrock-guardrailAction': 'INTERVENED'}}

In [43]:
test_guardrail(
    "Can you check the status of my policy POL-2024-AB-789456? "
    "I want to make sure it's still active before I file a claim.",
    label="Policy number — should ANONYMIZE"
)


TEST: Policy number — should ANONYMIZE
QUERY: Can you check the status of my policy POL-2024-AB-789456? I want to make sure it...

✅ PASSED — No intervention

RESPONSE: I'd be happy to help you check on your policy status, but I don't have access to your specific policy information or our customer database in this chat.

To check the status of policy **{Policy Number}**, you can:

1. **Log into your online account** on our website - you'll see your policy status an...

Tokens — Input: 41, Output: 171


{'action': 'NONE',
 'response': "I'd be happy to help you check on your policy status, but I don't have access to your specific policy information or our customer database in this chat.\n\nTo check the status of policy **{Policy Number}**, you can:\n\n1. **Log into your online account** on our website - you'll see your policy status and details there\n2. **Call our customer service line** - they can immediately verify your policy status\n3. **Contact your insurance agent** directly if you have one assigned\n\nBefore filing a claim, it's definitely smart to confirm your policy is active. When you do reach out, have your policy number ready along with any identification they may need to verify your account.\n\nIs there anything general about the claims process I can help explain while you're here?",
 'full_result': {'id': 'msg_bdrk_011QF2rYt4SoAdDRanga1u8e',
  'model': 'claude-sonnet-4-5-20250929',
  'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'

In [44]:
test_guardrail(
    "I need an update on claim CLM-ON-2026-004521. The adjuster was supposed "
    "to call me last week.",
    label="Claim reference — should ANONYMIZE"
)


TEST: Claim reference — should ANONYMIZE
QUERY: I need an update on claim CLM-ON-2026-004521. The adjuster was supposed to call ...

✅ PASSED — No intervention

RESPONSE: I'd be happy to help you get an update on claim {Claim Reference}.

To look into this for you, I'll need to verify a few details:

1. **Your name** and relationship to the claim (policyholder, claimant, etc.)
2. **Contact information** on file
3. **Policy number** associated with this claim

Regardi...

Tokens — Input: 35, Output: 173


{'action': 'NONE',
 'response': "I'd be happy to help you get an update on claim {Claim Reference}.\n\nTo look into this for you, I'll need to verify a few details:\n\n1. **Your name** and relationship to the claim (policyholder, claimant, etc.)\n2. **Contact information** on file\n3. **Policy number** associated with this claim\n\nRegarding the missed call from the adjuster - I apologize for that inconvenience. Once I verify your information, I can:\n- Check the current status of your claim\n- Review any notes from the adjuster\n- Ensure you get a callback scheduled, or\n- Connect you directly with the adjuster or their supervisor if available\n\nCould you please provide those verification details so I can assist you further?",
 'full_result': {'id': 'msg_bdrk_013U1mnMuJxxXsx2rcUJNGZw',
  'model': 'claude-sonnet-4-5-20250929',
  'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'd be happy to help you get an update on claim {Claim Reference}.\n\nTo

In [45]:
test_guardrail(
    "I need an update on claim CLM-ON-2026-004521. The adjuster was supposed "
    "to call me last week.",
    label="Claim reference — should ANONYMIZE"
)


TEST: Claim reference — should ANONYMIZE
QUERY: I need an update on claim CLM-ON-2026-004521. The adjuster was supposed to call ...

✅ PASSED — No intervention

RESPONSE: I'd be happy to help you get an update on claim {Claim Reference}.

To look into this for you, I'll need to verify a few details:

1. **Your name** and relationship to the claim (policyholder, claimant, etc.)
2. **Contact information** on file
3. **Policy number** (if available)

Regarding the adjus...

Tokens — Input: 35, Output: 227


{'action': 'NONE',
 'response': "I'd be happy to help you get an update on claim {Claim Reference}.\n\nTo look into this for you, I'll need to verify a few details:\n\n1. **Your name** and relationship to the claim (policyholder, claimant, etc.)\n2. **Contact information** on file\n3. **Policy number** (if available)\n\nRegarding the adjuster's call - I apologize for the inconvenience. Let me check the status of your claim and see what's in the notes about the scheduled contact.\n\nCould you also let me know:\n- When specifically were you expecting the call?\n- What was the call supposed to be regarding (estimate review, settlement discussion, additional information needed, etc.)?\n\nOnce I verify your information, I can:\n- Check the current status of your claim\n- Review the adjuster's notes\n- Either connect you directly with your adjuster or ensure you receive a callback today\n\nWhat's the best contact number to reach you if we need to follow up?",
 'full_result': {'id': 'msg_bdrk

In [46]:
test_guardrail(
    "What documents do I need to file a home insurance claim after a kitchen fire?",
    label="Clean query — no PII present"
)


TEST: Clean query — no PII present
QUERY: What documents do I need to file a home insurance claim after a kitchen fire?

✅ PASSED — No intervention

RESPONSE: # Documents Needed for a Kitchen Fire Insurance Claim

## Immediate Documentation
- **Photos and videos** of all fire, smoke, and water damage (before any cleanup)
- **Fire department report** (request a copy from responding department)
- **Police report** (if fire is suspected to be criminal/arson)...

Tokens — Input: 23, Output: 376


{'action': 'NONE',
 'response': '# Documents Needed for a Kitchen Fire Insurance Claim\n\n## Immediate Documentation\n- **Photos and videos** of all fire, smoke, and water damage (before any cleanup)\n- **Fire department report** (request a copy from responding department)\n- **Police report** (if fire is suspected to be criminal/arson)\n\n## Your Insurance Policy Documents\n- **Insurance policy number** and contact information\n- **Copy of your homeowners insurance policy**\n- **Previous inspection reports** or appraisals (if available)\n\n## Property Documentation\n- **Proof of ownership** for damaged items (receipts, credit card statements, bank records)\n- **Photos of items** before the fire (check phone, social media, family photos)\n- **Home inventory list** (if you maintained one)\n- **Repair estimates** from licensed contractors\n- **Receipts for temporary repairs** or emergency mitigation\n\n## Financial Records\n- **Receipts for additional living expenses** (hotel, meals, clo

In [47]:
test_guardrail(
    "What documents do I need to file a home insurance claim after a kitchen fire?",
    label="Clean query — no PII present"
)


TEST: Clean query — no PII present
QUERY: What documents do I need to file a home insurance claim after a kitchen fire?

✅ PASSED — No intervention

RESPONSE: # Documents Needed for a Kitchen Fire Insurance Claim

## Immediate Documentation
- **Photos and videos** of all fire damage (before any cleanup)
- **Fire department report** (request a copy from the responding station)
- **Police report** (if required by your insurer)

## Insurance Company Document...

Tokens — Input: 23, Output: 303


{'action': 'NONE',
 'response': "# Documents Needed for a Kitchen Fire Insurance Claim\n\n## Immediate Documentation\n- **Photos and videos** of all fire damage (before any cleanup)\n- **Fire department report** (request a copy from the responding station)\n- **Police report** (if required by your insurer)\n\n## Insurance Company Documents\n- **Completed claim form** (from your insurer)\n- **Your insurance policy** (policy number and coverage details)\n- **Proof of ownership** of your home\n\n## Damage Documentation\n- **Detailed inventory** of damaged/destroyed items with:\n  - Descriptions and approximate ages\n  - Original purchase prices (if known)\n  - Receipts, photos, or videos showing items before damage\n- **Repair estimates** from licensed contractors\n- **Receipts for emergency repairs** you've already made\n\n## Financial Records\n- **Receipts for additional living expenses** (hotel, meals, etc.)\n- **Receipts for emergency purchases** (clothing, toiletries)\n- **Records of

Wrap Up — PII detection with built-in detectors and custom regex

Implements 10 built-in PII detectors (SSN, SIN, email, phone, credit
card, etc.) and 3 custom regex patterns for insurance identifiers
(policy numbers, claim references, agent codes). Configured BLOCK for
high-sensitivity types, ANONYMIZE for functional identifiers.